# Step 1B: Cell-Type Deconvolution

## Overview
This notebook estimates cell-type-specific protein abundances in bulk ROSMAP samples using a single-nucleus RNA-seq reference (Mathys et al. 2019). Deconvolution translates bulk signal into cell-type-resolved proportions and profiles.

## Scientific Rationale
Bulk proteomics is a mixture of contributions from multiple cell types. By using a **reference signature matrix** (mean protein expression per cell type from snRNA-seq), we solve a constrained optimization problem to estimate:
1. **Cell-type proportions**: What % of each sample is Ex, In, Ast, etc.
2. **Cell-type-specific profiles**: Estimated expression of each protein in each cell type

This enables cell-type-specific disease mechanisms to be discovered.

## Key Inputs
- `data/raw/mathys_reference.h5ad`: Mathys 2019 snRNA-seq reference (Seurat object in HDF5 format)
- `data/processed/rosmap_proteomics_cleaned.csv`: Clean bulk proteomics from Step 1A

## Expected Outputs
- `data/processed/cell_type_proportions.csv`: (n_patients × 6) matrix with cell-type fractions
- `data/processed/deconvolved_profiles.csv`: Cell-type-specific abundance estimates
- `results/step1/cell_type_proportions.png`: Stacked bar chart
- `results/step1/celltype_proportion_comparison.png`: Violin plots (AD vs Control)

## Method: Non-Negative Least Squares (NNLS)
For each bulk sample: minimize ||bulk - Reference @ proportions||² subject to proportions ≥ 0 and sum = 1

## Execution Time
- **Full data**: ~20-30 minutes (linear in n_samples)
- **Test mode**: ~3 seconds

## Success Criteria
- ✓ All proportions sum to 1 (normalized)
- ✓ Proportions in [0, 1] (valid probability range)
- ✓ Statistical differences found between AD/Control (or None if no signal)
- ✓ Visualizations generated without errors

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import mannwhitneyu
import scanpy as sc
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

### Library Overview
- **pandas/numpy**: Data manipulation and numerical operations
- **scipy.optimize.minimize**: Constrained optimization (for NNLS)
- **scipy.stats.mannwhitneyu**: Non-parametric statistical test for group comparisons
- **scanpy**: Single-cell/single-nucleus analysis (reads AnnData .h5ad format)
- **matplotlib/seaborn**: Publication-quality visualization

## Configuration

In [ ]:
# Data paths
RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"
RESULTS_DIR = "../results/step1"

# Ensure output directories exist
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Cell types in Mathys reference
CELL_TYPES = ['Ex', 'In', 'Ast', 'Oli', 'Mic', 'OPCs']

print("Configuration loaded.")

## Generate Synthetic Data (if needed)
This cell generates fake Mathys reference and proteomics data for testing.

In [ ]:
# Run this cell if you don't have real data yet
import os
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print("[*] Generating synthetic Mathys 2019 reference...")

np.random.seed(42)

# Create synthetic snRNA-seq reference
n_cells = 8066
n_genes = 10000
n_subjects = 48
cell_types = ['Ex', 'In', 'Ast', 'Oli', 'Mic', 'OPCs']
cell_type_proportions = [0.35, 0.15, 0.20, 0.15, 0.10, 0.05]

# Assign cell types and subjects
cell_type_labels = np.random.choice(cell_types, size=n_cells, p=cell_type_proportions)
subject_ids = np.random.choice([f'ROSMAP_{i:03d}' for i in range(n_subjects)], size=n_cells)

# Gene names
gene_names = [f'GENE_{i}' for i in range(n_genes)]

# Expression matrix with cell-type-specific patterns
X_counts = np.zeros((n_cells, n_genes))
for ct in cell_types:
    mask = cell_type_labels == ct
    n_ct_cells = mask.sum()
    ct_baseline = np.random.exponential(scale=0.5, size=n_genes)
    ct_noise = np.random.normal(0, 1, size=(n_ct_cells, n_genes))
    X_counts[mask, :] = np.maximum(ct_baseline[np.newaxis, :] + ct_noise, 0)

# Create AnnData object
adata = sc.AnnData(
    X=X_counts,
    obs=pd.DataFrame({
        'cell_type': cell_type_labels,
        'broad.cell.type': cell_type_labels,
        'subject_id': subject_ids
    }, index=[f'cell_{i}' for i in range(n_cells)]),
    var=pd.DataFrame({'gene_id': gene_names}, index=gene_names)
)

# Save reference
adata.write(f'{RAW_DATA_DIR}/mathys_reference.h5ad')
print(f"  Saved: {RAW_DATA_DIR}/mathys_reference.h5ad")
print(f"  Shape: {adata.shape}")
print(f"  Cell types: {adata.obs['cell_type'].value_counts().to_dict()}")

# Generate synthetic bulk proteomics
print("\n[*] Generating synthetic bulk proteomics...")

n_patients = 180
n_proteins = 1500

# Select proteins that overlap with reference
protein_indices = np.random.choice(n_genes, size=n_proteins, replace=False)
selected_genes = [gene_names[i] for i in protein_indices]

# Compute cell-type reference profiles
reference_means = np.zeros((len(cell_types), n_genes))
for i, ct in enumerate(cell_types):
    ct_mask = adata.obs['cell_type'] == ct
    reference_means[i, :] = adata.X[ct_mask, :].mean(axis=0)

# Generate bulk samples (weighted sum of cell-type profiles)
bulk_data = np.zeros((n_patients, n_proteins))
ground_truth_proportions = []

for p in range(n_patients):
    # Random cell-type proportions
    proportions = np.random.dirichlet([1.0] * len(cell_types))
    ground_truth_proportions.append(proportions)
    
    # Convolve cell-type profiles
    bulk_profile = proportions @ reference_means[np.ix_(np.arange(len(cell_types)), protein_indices)]
    
    # Add noise
    noise = np.random.normal(0, 0.1, size=n_proteins)
    bulk_data[p, :] = np.maximum(bulk_profile + noise, 0)

# Save bulk data
bulk_df = pd.DataFrame(
    bulk_data,
    index=[f'patient_{i:03d}' for i in range(n_patients)],
    columns=selected_genes
)
bulk_df.index.name = 'patient_id'
bulk_df.to_csv(f'{PROCESSED_DATA_DIR}/rosmap_proteomics_cleaned.csv')
print(f"  Saved: {PROCESSED_DATA_DIR}/rosmap_proteomics_cleaned.csv")
print(f"  Shape: {bulk_df.shape}")

# Save metadata
diagnoses = np.random.choice(['Control', 'AD'], size=n_patients, p=[0.55, 0.45])
metadata_df = pd.DataFrame({
    'diagnosis': diagnoses,
    'age_death': np.random.randint(60, 100, n_patients),
    'msex': np.random.choice([0, 1], n_patients),
    'pmi': np.random.uniform(2, 30, n_patients)
}, index=bulk_df.index)
metadata_df.index.name = 'projid'
metadata_df.to_csv(f'{PROCESSED_DATA_DIR}/rosmap_metadata.csv')
print(f"  Saved: {PROCESSED_DATA_DIR}/rosmap_metadata.csv")

print("\n[✓] Synthetic data generation complete!")

## Helper Functions

In [ ]:
def load_mathys_reference(file_path):
    """
    Load Mathys et al. 2019 snRNA-seq reference.
    Handles .h5ad format (AnnData).
    """
    print("[1] Loading Mathys 2019 snRNA-seq reference...")
    
    if file_path.endswith('.h5ad'):
        adata = sc.read_h5ad(file_path)
        print(f"  Loaded .h5ad: {adata.shape}")
    else:
        raise ValueError(f"Unsupported format: {file_path}. Use .h5ad")
    
    print(f"  Variables (genes): {adata.shape[1]}")
    print(f"  Observations (cells): {adata.shape[0]}")
    
    return adata

In [ ]:
def load_bulk_proteomics(file_path):
    """
    Load bulk proteomics matrix (n_samples x n_proteins).
    """
    print("[2] Loading bulk proteomics data...")
    
    df = pd.read_csv(file_path, index_col=0)
    print(f"  Shape: {df.shape[0]} samples x {df.shape[1]} proteins")
    print(f"  Missing values: {df.isnull().sum().sum()}")
    
    return df

In [ ]:
def load_metadata(file_path):
    """
    Load clinical metadata.
    """
    print("[3] Loading clinical metadata...")
    
    df = pd.read_csv(file_path, index_col=0)
    print(f"  Shape: {df.shape}")
    print(f"  Diagnosis counts: {df['diagnosis'].value_counts().to_dict()}")
    
    return df

In [ ]:
def extract_cell_type_reference(adata, cell_type_col='cell_type', cell_types=None):
    """
    Build reference matrix: mean expression per cell type.
    Returns: (cell_type_profiles, cell_types, gene_names)
        - cell_type_profiles: (n_cell_types x n_genes)
    """
    print("[4] Building cell-type reference profiles...")
    
    if cell_types is None:
        cell_types = adata.obs[cell_type_col].unique()
    
    gene_names = np.array(adata.var.index)
    n_genes = len(gene_names)
    
    # Compute mean expression per cell type
    reference_profiles = np.zeros((len(cell_types), n_genes))
    for i, ct in enumerate(cell_types):
        mask = adata.obs[cell_type_col] == ct
        n_cells = mask.sum()
        
        # Extract expression and compute mean
        if hasattr(adata.X, 'toarray'):  # sparse matrix
            reference_profiles[i, :] = adata.X[mask, :].toarray().mean(axis=0)
        else:  # dense array
            reference_profiles[i, :] = adata.X[mask, :].mean(axis=0)
        
        print(f"  {ct}: {n_cells} cells, mean expression computed")
    
    return reference_profiles, cell_types, gene_names

In [ ]:
def nnls_deconvolve(bulk_row, reference_profiles):
    """
    Non-negative least squares deconvolution.
    
    Problem formulation:
      Given: bulk_row (observed bulk expression vector)
             reference_profiles (cell-type-specific mean expression)
      
      Find: proportions that minimize squared error
      argmin_x ||bulk_row - reference @ proportions||²
      
    Constraints:
      - x_i >= 0 (proportions non-negative)
      - sum(x_i) = 1 (proportions sum to 1, i.e., probabilities)
    
    Why NNLS?: Unlike unconstrained least squares, NNLS ensures
    interpretable cell-type proportions (can't have negative fractions).
    
    Args:
        bulk_row: (n_features,) bulk expression vector
        reference_profiles: (n_cell_types, n_features) reference matrix
    
    Returns:
        proportions: (n_cell_types,) normalized cell-type proportions
    """
    # Define objective function: sum of squared residuals
    def objective(x):
        # x are the cell-type proportions we're solving for
        # reference_profiles.T @ x gives predicted bulk expression
        residuals = bulk_row - reference_profiles.T @ x
        return np.sum(residuals ** 2)
    
    # Constraint: sum(proportions) = 1 (probabilities)
    def sum_constraint(x):
        return np.sum(x) - 1.0
    
    # Initial guess: equal proportions (uniform distribution)
    x0 = np.ones(reference_profiles.shape[0]) / reference_profiles.shape[0]
    
    # Bounds: each proportion in [0, 1]
    bounds = [(0, 1) for _ in range(reference_profiles.shape[0])]
    
    # Constraint dictionary for SLSQP optimizer
    constraints = {'type': 'eq', 'fun': sum_constraint}
    
    # Optimize using SLSQP (Sequential Least Squares Programming)
    # This is a gradient-based optimizer that handles constraints
    result = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)
    
    # Normalize to ensure sum = 1 (handles numerical precision issues)
    proportions = result.x / result.x.sum()
    
    return proportions

In [ ]:
def nnls_deconvolve(bulk_row, reference_profiles):
    """
    Non-negative least squares deconvolution.
    Solves: argmin ||bulk_row - reference @ proportions||^2
    subject to: proportions >= 0, sum(proportions) = 1
    
    Args:
        bulk_row: (n_features,) bulk expression vector
        reference_profiles: (n_cell_types, n_features) reference matrix
    
    Returns:
        proportions: (n_cell_types,) cell-type proportions summing to 1
    """
    # Objective: minimize squared error
    def objective(x):
        residuals = bulk_row - reference_profiles.T @ x
        return np.sum(residuals ** 2)
    
    # Constraint: sum = 1 (handled via constraint dictionary)
    def sum_constraint(x):
        return np.sum(x) - 1.0
    
    # Initial guess: equal proportions
    x0 = np.ones(reference_profiles.shape[0]) / reference_profiles.shape[0]
    
    # Bounds: 0 <= x_i <= 1
    bounds = [(0, 1) for _ in range(reference_profiles.shape[0])]
    
    # Constraint: sum = 1
    constraints = {'type': 'eq', 'fun': sum_constraint}
    
    # Optimize
    result = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)
    
    # Ensure sum to 1
    proportions = result.x / result.x.sum()
    
    return proportions

In [ ]:
def run_deconvolution(bulk_df, reference_profiles, cell_types):
    """
    Run deconvolution on all bulk samples.
    Returns: (cell_type_proportions, deconvolved_profiles)
        - cell_type_proportions: (n_samples x n_cell_types)
        - deconvolved_profiles: (n_samples x n_proteins x n_cell_types)
    """
    print("[6] Running NNLS deconvolution...")
    
    n_samples = bulk_df.shape[0]
    n_proteins = bulk_df.shape[1]
    n_cell_types = reference_profiles.shape[0]
    
    # Initialize outputs
    cell_type_proportions = np.zeros((n_samples, n_cell_types))
    deconvolved_profiles = np.zeros((n_samples, n_proteins, n_cell_types))
    
    # Deconvolve each sample
    for i, (sample_id, bulk_row) in enumerate(bulk_df.iterrows()):
        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/{n_samples} samples...")
        
        # Estimate proportions
        proportions = nnls_deconvolve(bulk_row.values, reference_profiles)
        cell_type_proportions[i, :] = proportions
        
        # Estimate cell-type-specific profiles
        # For each protein: profile_ij = reference_j * proportion_i
        for j in range(n_cell_types):
            deconvolved_profiles[i, :, j] = reference_profiles[j, :] * proportions[j]
    
    print(f"  Completed deconvolution for {n_samples} samples")
    
    return cell_type_proportions, deconvolved_profiles

In [ ]:
def save_deconvolution_results(bulk_df, cell_type_proportions, 
                               deconvolved_profiles, cell_types, 
                               processed_dir):
    """
    Save deconvolution results.
    """
    print("[7] Saving deconvolution results...")
    
    # Save cell-type proportions
    proportions_df = pd.DataFrame(
        cell_type_proportions,
        index=bulk_df.index,
        columns=cell_types
    )
    proportions_file = f"{processed_dir}/cell_type_proportions.csv"
    proportions_df.to_csv(proportions_file)
    print(f"  Saved: {proportions_file}")
    print(f"  Shape: {proportions_df.shape}")
    
    # Save deconvolved profiles (multi-index DataFrame)
    deconvolved_list = []
    for i, sample_id in enumerate(bulk_df.index):
        for j, protein_id in enumerate(bulk_df.columns):
            for k, ct in enumerate(cell_types):
                deconvolved_list.append({
                    'sample_id': sample_id,
                    'protein_id': protein_id,
                    'cell_type': ct,
                    'abundance': deconvolved_profiles[i, j, k]
                })
    
    deconvolved_df = pd.DataFrame(deconvolved_list)
    deconvolved_file = f"{processed_dir}/deconvolved_profiles.csv"
    deconvolved_df.to_csv(deconvolved_file, index=False)
    print(f"  Saved: {deconvolved_file}")
    print(f"  Shape: {deconvolved_df.shape}")
    
    return proportions_df, deconvolved_df

In [ ]:
def plot_cell_type_proportions(proportions_df, metadata_df, results_dir):
    """
    Stacked bar chart of cell-type proportions, sorted by diagnosis.
    Each bar is one patient, colored segments show cell-type proportions.
    """
    print("[8] Generating cell-type proportions visualization...")
    
    # Add diagnosis to proportions
    data = proportions_df.copy()
    data['diagnosis'] = metadata_df.loc[data.index, 'diagnosis'].values
    
    # Sort by diagnosis, then by first cell type proportion for visual grouping
    data = data.sort_values(by=['diagnosis', data.columns[0]])
    
    # Create stacked bar chart
    fig, ax = plt.subplots(figsize=(16, 6))
    
    # Get cell types (all except 'diagnosis')
    cell_types = [c for c in data.columns if c != 'diagnosis']
    
    # Colors for cell types
    colors = plt.cm.Set3(np.linspace(0, 1, len(cell_types)))
    
    # Plot stacked bars
    bottom = np.zeros(len(data))
    for i, ct in enumerate(cell_types):
        ax.bar(range(len(data)), data[ct], bottom=bottom, label=ct, color=colors[i], alpha=0.8)
        bottom += data[ct].values
    
    # Add diagnosis separator line
    diagnosis_change = data['diagnosis'].ne(data['diagnosis'].shift()).cumsum()
    for pos in np.where(diagnosis_change.diff().fillna(0) != 0)[0]:
        ax.axvline(x=pos - 0.5, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    ax.set_xlabel('Patients (sorted by diagnosis)', fontsize=12)
    ax.set_ylabel('Cell-Type Proportion', fontsize=12)
    ax.set_title('Estimated Cell-Type Proportions per Patient', fontsize=13, fontweight='bold')
    ax.legend(title='Cell Type', loc='upper left', fontsize=9)
    ax.set_ylim([0, 1])
    plt.xticks([])
    
    # Save
    fig_file = f"{results_dir}/cell_type_proportions.png"
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {fig_file}")
    plt.close()

In [ ]:
def plot_celltype_comparison(proportions_df, metadata_df, results_dir):
    """
    Violin plots: distribution of each cell-type proportion in AD vs control.
    Includes Mann-Whitney U test p-values.
    """
    print("[9] Generating cell-type comparison visualization...")
    
    # Prepare data
    data = proportions_df.copy()
    data['diagnosis'] = metadata_df.loc[data.index, 'diagnosis'].values
    
    cell_types = proportions_df.columns.tolist()
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    # For each cell type
    for idx, ct in enumerate(cell_types):
        ax = axes[idx]
        
        # Prepare data for violin plot
        plot_data = []
        for diagnosis in ['Control', 'AD']:
            values = data[data['diagnosis'] == diagnosis][ct].values
            plot_data.append(values)
        
        # Create violin plot
        parts = ax.violinplot(plot_data, positions=[0, 1], showmeans=True, showmedians=True)
        
        # Mann-Whitney U test
        stat, pval = mannwhitneyu(plot_data[0], plot_data[1])
        
        # Styling
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Control', 'AD'])
        ax.set_ylabel('Proportion', fontsize=11)
        ax.set_title(f'{ct}\n(p={pval:.2e})', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim([data[ct].min() - 0.05, data[ct].max() + 0.05])
    
    plt.tight_layout()
    fig_file = f"{results_dir}/celltype_proportion_comparison.png"
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {fig_file}")
    plt.close()

In [ ]:
## Main Pipeline Execution

This cell orchestrates:
1. Load Mathys reference and bulk data
2. Extract cell-type profiles from reference
3. Match overlapping features (genes in reference must overlap with proteins in bulk)
4. Run NNLS deconvolution on each sample (fast: ~0.1s/sample)
5. Generate visualization comparing cell-type proportions between AD and Control

**Expected output**:
- Cell-type proportions table (checks: sum to 1, all in [0, 1])
- Statistical tests comparing AD vs Control (Mann-Whitney U)
- Visualizations showing disease-associated cell-type changes
- Ready signal for Step 1C (Pseudotime)

## Main Pipeline

In [ ]:
print("\n" + "="*70)
print("STEP 1B: CELL-TYPE DECONVOLUTION")
print("="*70 + "\n")

try:
    # Load data
    adata_ref = load_mathys_reference(f"{RAW_DATA_DIR}/mathys_reference.h5ad")
    bulk_df = load_bulk_proteomics(f"{PROCESSED_DATA_DIR}/rosmap_proteomics_cleaned.csv")
    metadata_df = load_metadata(f"{PROCESSED_DATA_DIR}/rosmap_metadata.csv")
    
    # Extract cell-type profiles from reference
    ref_profiles, cell_types_list, ref_genes = extract_cell_type_reference(
        adata_ref, 
        cell_type_col='cell_type',
        cell_types=CELL_TYPES
    )
    
    # Match genes and proteins
    ref_idx, bulk_idx, overlapping_features = match_genes_to_proteins(
        ref_genes, 
        bulk_df.columns.values
    )
    
    # Subset data to overlapping features
    ref_profiles_matched = ref_profiles[:, ref_idx]
    bulk_df_matched = bulk_df.iloc[:, bulk_idx]
    
    print(f"\n  Using {len(overlapping_features)} overlapping features for deconvolution")
    
    # Run deconvolution
    cell_type_proportions, deconvolved_profiles = run_deconvolution(
        bulk_df_matched, 
        ref_profiles_matched, 
        CELL_TYPES
    )
    
    # Save results
    proportions_df, deconvolved_df = save_deconvolution_results(
        bulk_df_matched,
        cell_type_proportions,
        deconvolved_profiles,
        CELL_TYPES,
        PROCESSED_DATA_DIR
    )
    
    # Visualizations
    plot_cell_type_proportions(proportions_df, metadata_df, RESULTS_DIR)
    plot_celltype_comparison(proportions_df, metadata_df, RESULTS_DIR)
    
    # Statistical testing
    significant_changes = test_significant_changes(proportions_df, metadata_df)
    
    print("\n" + "="*70)
    print("DECONVOLUTION COMPLETE ✓")
    print("="*70)
    print(f"\nOutputs saved to: {PROCESSED_DATA_DIR}")
    print(f"Figures saved to: {RESULTS_DIR}")
    print("\nReady for Step 1C: Pseudotime & NMF Analysis\n")

except Exception as e:
    print(f"\n❌ ERROR: {str(e)}")
    import traceback
    traceback.print_exc()